# Конкурентный анализ АГАТ

## tl;dr

- Детально оценены **12 конкурентов** в шести продуктовых кластерах; UiPath и DIY-стек учтены как соседние альтернативы вне скоринга.
- Текущая функциональная широта АГАТ — **58,3%**, тогда как лидеры набирают **85–86,7%**. Разрыв особенно велик в RAG, интеграциях/MCP, eval/replay и production scale.
- АГАТ — единственный продукт в подтверждённой выборке со score **4/4** за централизованное управление разнородным outbound-only парком локальных моделей. Это конкурентный клин, но пока не полноценный moat без table-stakes.
- Аудитория конкурентов опережает АГАТ на **2–4 стадии зрелости** по proxy-шкале; это не market share и не число активных пользователей.
- Сценарий P0 поднимает соответствие выбранной sovereign-fleet стратегии с **65,0% до 79,25%**; P1 — до **90,0%**. Это сценарный скоринг, не прогноз срока или продаж.

## Context & Methods

**Вопрос решения:** какие возможности добавлять, как позиционировать АГАТ, кого считать целевой аудиторией и насколько зрелые конкуренты опережают продукт.

**Период:** срез на 2026-08-14. Возможности АГАТ подтверждены по репозиторию и документации; конкуренты — по официальной документации, сайтам и vendor-owned GitHub. Оцениваются только документированные текущие возможности, не roadmap.

**Feature score 0–4:** 0 — не подтверждено; 1 — базово/DIY/preview; 2 — функционально, но ограниченно; 3 — зрелая возможность; 4 — сильная или category-leading реализация.

**Audience maturity 1–5:** 1 — MVP/нет данных о спросе; 2 — ранняя ниша; 3 — заметная нишевая/OSS-аудитория; 4 — широкое production/enterprise использование; 5 — платформенная/глобальная дистрибуция. Это качественный proxy по официальным adoption claims, экосистеме и production maturity, а не market share.

Два веса проверяют чувствительность вывода: `sovereign` отражает выбранный клин АГАТ; `enterprise` — типичный рынок enterprise automation. Unweighted breadth показывает чистую широту каталога возможностей.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd()
if not (DATA_DIR / 'competitive-analysis-scores.csv').exists():
    DATA_DIR = DATA_DIR / 'docs'

scores = pd.read_csv(DATA_DIR / 'competitive-analysis-scores.csv')
categories = pd.read_csv(DATA_DIR / 'competitive-analysis-categories.csv')
profiles = pd.read_csv(DATA_DIR / 'competitive-analysis-profiles.csv')
sources = pd.read_csv(DATA_DIR / 'competitive-analysis-sources.csv')
roadmap = pd.read_csv(DATA_DIR / 'competitive-analysis-roadmap.csv')
audiences = pd.read_csv(DATA_DIR / 'competitive-analysis-audiences.csv')
scenarios = pd.read_csv(DATA_DIR / 'competitive-analysis-scenarios.csv')

feature_ids = categories['category_id'].tolist()
assert len(feature_ids) == 15 and categories['category_id'].is_unique
assert categories['weight_sovereign'].sum() == 100
assert categories['weight_enterprise'].sum() == 100
assert scores['product'].is_unique and len(scores) == 13
assert scores[feature_ids].ge(0).all().all() and scores[feature_ids].le(4).all().all()
assert scores['audience_maturity'].between(1, 5).all()
assert set(profiles['product']) == set(scores['product'])
known_sources = set(sources['source_id'])
used_sources = {sid for group in profiles['source_ids'] for sid in group.split('|')}
assert used_sources <= known_sources
print(f'Loaded {len(scores)-1} competitors, {len(feature_ids)} feature categories, {len(sources)} evidence records.')

## Data

Категории и веса. Sovereign-веса намеренно выделяют distributed fleet и resource routing; enterprise-веса — integrations, durability, governance и scale.

In [ ]:
categories[['category_ru', 'definition', 'weight_sovereign', 'weight_enterprise']]

In [ ]:
rubric = scores[['product', 'comparison_type', *feature_ids, 'audience_maturity']].copy()
rubric

## Results

Функциональная широта — среднее 15 оценок, приведённое к 100%. Strategic fit — взвешенный score, где 100% означает 4/4 во всех категориях. Feature gap сравнивается с текущим АГАТ.

In [ ]:
w_sovereign = categories.set_index('category_id')['weight_sovereign']
w_enterprise = categories.set_index('category_id')['weight_enterprise']
ranked = scores.copy()
ranked['breadth_pct'] = ranked[feature_ids].mean(axis=1) * 25
ranked['sovereign_fit_pct'] = ranked[feature_ids].mul(w_sovereign, axis=1).sum(axis=1) / 4
ranked['enterprise_fit_pct'] = ranked[feature_ids].mul(w_enterprise, axis=1).sum(axis=1) / 4
agat = ranked.loc[ranked['product'].eq('АГАТ')].iloc[0]
ranked['feature_gap_pp'] = ranked['breadth_pct'] - agat['breadth_pct']
ranked['audience_lead_stages'] = ranked['audience_maturity'] - int(agat['audience_maturity'])

def distance_band(gap):
    if gap <= 0: return 'АГАТ не позади'
    if gap <= 5: return 'Паритет'
    if gap <= 15: return 'Умеренный разрыв'
    if gap <= 25: return 'Значительный разрыв'
    return 'Большой разрыв'

ranked['feature_distance'] = ranked['feature_gap_pp'].map(distance_band)
view = ranked[['product','comparison_type','breadth_pct','sovereign_fit_pct','enterprise_fit_pct','feature_gap_pp','feature_distance','audience_maturity','audience_lead_stages']].sort_values('breadth_pct', ascending=False)
view

In [ ]:
leaders = {
    'breadth': ranked.sort_values('breadth_pct', ascending=False).iloc[0]['product'],
    'sovereign': ranked.sort_values('sovereign_fit_pct', ascending=False).iloc[0]['product'],
    'enterprise': ranked.sort_values('enterprise_fit_pct', ascending=False).iloc[0]['product'],
}
competitors = scores.loc[~scores['product'].eq('АГАТ')]
checks = pd.DataFrame([
    {'check': 'Scored competitors', 'value': len(competitors)},
    {'check': 'Competitors with RAG score >= 2', 'value': int(competitors['rag_memory'].ge(2).sum())},
    {'check': 'Competitors with integration score >= 3', 'value': int(competitors['integrations_mcp'].ge(3).sum())},
    {'check': 'Competitors with eval score >= 2', 'value': int(competitors['evaluation'].ge(2).sum())},
    {'check': 'Products other than AGAT with fleet score > 0', 'value': int(competitors['distributed_fleet'].gt(0).sum())},
    {'check': 'AGAT feature breadth', 'value': f"{agat['breadth_pct']:.1f}%"},
    {'check': 'Breadth leader', 'value': leaders['breadth']},
    {'check': 'Sovereign-fit leader today', 'value': leaders['sovereign']},
    {'check': 'Enterprise-fit leader', 'value': leaders['enterprise']},
])
checks

### Sensitivity and roadmap scenarios

Рейтинг меняется при смене job-to-be-done: это важнее одного универсального leaderboard. Ниже — текущие продукты и сценарии АГАТ. P0/P1 scores показывают, как изменится полнота при успешной реализации конкретных capabilities; они не учитывают сроки, качество UX, продажи и execution risk.

In [ ]:
scenario_scores = scenarios.copy()
scenario_scores['breadth_pct'] = scenario_scores[feature_ids].mean(axis=1) * 25
scenario_scores['sovereign_fit_pct'] = scenario_scores[feature_ids].mul(w_sovereign, axis=1).sum(axis=1) / 4
scenario_scores['enterprise_fit_pct'] = scenario_scores[feature_ids].mul(w_enterprise, axis=1).sum(axis=1) / 4
scenario_scores[['scenario','description','breadth_pct','sovereign_fit_pct','enterprise_fit_pct']]

In [ ]:
sensitivity = pd.DataFrame({
    'Breadth rank': ranked.sort_values('breadth_pct', ascending=False)['product'].head(6).tolist(),
    'Sovereign-fit rank': ranked.sort_values('sovereign_fit_pct', ascending=False)['product'].head(6).tolist(),
    'Enterprise-fit rank': ranked.sort_values('enterprise_fit_pct', ascending=False)['product'].head(6).tolist(),
}, index=range(1, 7))
sensitivity.index.name = 'Rank'
sensitivity

## Takeaways

1. **Не позиционировать АГАТ как ещё один visual AI builder.** В этой категории Dify, Flowise, Sim, Langflow и Just AI уже шире; n8n и Copilot дополнительно выигрывают дистрибуцией.
2. **Позиционирование:** «Control plane для внутренних AI-процессов на локальных моделях и разнородном железе — без публикации model endpoints».
3. **Beachhead:** регулируемые организации и интеграторы с 2–20 локальными GPU/CPU узлами, approvals/audit и чувствительными данными. Generic SMB automation и массовые чатботы — не первичный рынок.
4. **P0:** local RAG, MCP gateway/connector SDK, schedules/webhooks и eval/replay. Это закрывает table-stakes, не разрушая клин.
5. **P1 moat:** benchmark-aware router, risk-tier policy engine, signed fleet lifecycle и production data plane.
6. **Audience gap — отдельная проблема:** даже после feature parity понадобятся reference deployments, paid pilots, partner channel и доказательства reliability/security.

**Caveats:** scoring основан на публичной документации и экспертной рубрике; отсутствие evidence снижает score, но не доказывает физическое отсутствие функции. Vendor adoption claims не верифицированы независимо. Цены, UX usability, implementation quality, revenue и active users не сравнивались.